Idea: 

1. Function create_lstm_model should take in a dict of parameters to define a LSTM Model Architecture
   * this can be very basic but should provide a coverage of options 
2. Function which tunes the hyperparameters
3. run model and log everything using mlflow 

In [1]:
import os
import numpy as np
import pandas as pd 
import itertools
import mlflow
from keras.optimizers import Adam
from IPython.display import clear_output

import matplotlib.pyplot as plt

from keras.callbacks import EarlyStopping

from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import explained_variance_score, mean_absolute_error, r2_score, mean_squared_error

# custom functions for feature engineering
from helper_functions import * 

from keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

2023-12-03 12:51:45.903256: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-03 12:51:45.903288: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-03 12:51:45.903321: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-03 12:51:46.039529: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0. Your GPU:
  NVIDIA GeForce GTX 1060 6GB, compute capability 6.1
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once


2023-12-03 12:51:49.796392: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-12-03 12:51:49.932067: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-12-03 12:51:49.932269: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [2]:
# setting up credentials for storing the model
os.environ["AWS_ACCESS_KEY_ID"] = "eITEO5kyE7hccuy7UTHv"
os.environ["AWS_SECRET_ACCESS_KEY"] = "5KBCscit30Z70bSVGIMvMBBoqV8ydn232o2MW9RA"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://172.1.0.12:2000"

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [3]:
# this might not be perfect yet

def create_lstm_model(layers_config):
    model = Sequential()

    for layer_conf in layers_config:
        layer_type = layer_conf["type"]

        if layer_type == "LSTM":
            lstm_kwargs = {
                "units": layer_conf["units"],
                "return_sequences": layer_conf["return_sequences"]
            }
            # Add input_shape only for the first LSTM layer if specified
            if "input_shape" in layer_conf:
                lstm_kwargs["input_shape"] = layer_conf["input_shape"]

            model.add(LSTM(**lstm_kwargs))

        elif layer_type == "Dropout":
            model.add(Dropout(layer_conf["rate"]))

        elif layer_type == "Dense":
            model.add(Dense(units=layer_conf["units"], activation=layer_conf["activation"]))

    return model

In [4]:
features = ['is_holiday', 'close_price', 'volume', 'is_weekend', 'count', 'hour_cos',
            'hour_sin', 'day_of_week_cos', 'day_of_week_sin', 'month_cos', 'month_sin',
            'day_of_month_sin', 'day_of_month_cos', 'year_normalized', 'close_lag12',
             'ema_6_close_price', 'ema_12_close_price']

# , 'upper_bollinger_band', 'lower_bollinger_band'
# 'ema_6_close_price',
#'sma_12_close_price',
# generating all combinations of optional features
#'close_lag168',


param_grid = {
    'features': [features],
    'lookback': [16],
    'learning_rate': [0.001],
    'optimizer': ['adam'],
    'batch_size': [16],
    'datatype': [np.float32],
    'epochs': [50],
    'loss': ['mean_absolute_error'],
    'metrics' : [['mean_absolute_error', 'mean_squared_error', 'accuracy']] # double list to not alternate between the metrics
}

grid = ParameterGrid(param_grid)

len(grid)


1

In [5]:
def bollinger_bands(data, days=7):
    # calculating the rolling standard deviation
    std_dev = data['close_price'].rolling(window=days*24).std()
    data = simple_moving_average(data, window_sizes=[days*24]) # compute the moving average for a window of 10 days
    # Calculate the upper and lower Bollinger Bands
    colum_name = "sma_"+str(days*24)+"_close_price"
    # Calculate the upper and lower Bollinger Bands
    data['upper_bollinger_band'] = data[colum_name] + (std_dev * 2)
    data['lower_bollinger_band'] = data[colum_name] - (std_dev * 2)
    return data


In [6]:
def data_prep(data):
    # data prep
    data = data.drop(columns=['open_price', 'high', 'low'], axis=1)

    data = process_timestamp(data, fill=True)
    data = add_feature_date(data)
    data = add_holiday_feature(data)
    data = apply_cyclic_encoding(data, columms=['hour', 'day_of_week', 'month'])
    data = apply_day_of_month_encoding(data)
    data = normalize_year(data) # has to be updated yearly (because of the min-max scaler - max+1 is currently set = 2024)
    data['close_price_true'] = data['close_price'] # save the true price 
    data = apply_log_scaler(data, columns=['close_price', 'volume', 'count'])
    data = add_feature_lag(data, lags=[1,4,6,12,24,168])
    data = simple_moving_average(data, window_sizes=[6,12,168], columns=['close_price']) # window sizes are in hours
    data = exponential_moving_average(data, span_sizes=[6,12,24,96,168], columns=['close_price']) # span sizes are in hours
    data = bollinger_bands(data)
    # not needed as these values are encoded to be cyclic 
    data = data.drop(columns=['symbol_id', 'hour', 'day_of_week', 'day_of_month', 'month', 'year'], axis=1)

    # dropping nan values which are created because of simple_moving_average and lags 
    data = data.dropna()

    return data

In [7]:
# Prep data once for all models: 
file_path = 'kraken_ohlc_hour_count.csv'
raw_data = pd.read_csv(file_path)
data = data_prep(raw_data)

test = data.iloc[int(len(data)*0.99):] 
training = data.iloc[:int(len(data)*0.99)]
training.head()

,bucket,close_price,volume,count,is_weekend,is_holiday,hour_cos,hour_sin,day_of_week_cos,day_of_week_sin,...,sma_6_close_price,sma_12_close_price,sma_168_close_price,ema_6_close_price,ema_12_close_price,ema_24_close_price,ema_96_close_price,ema_168_close_price,upper_bollinger_band,lower_bollinger_band
168,2015-08-14 14:00:00+00:00,1.099205,3.931159,1.098612,0,0,-8.660254e-01,-0.500000,-0.900969,-0.433884,...,1.119724,1.099312,0.864821,1.113550,1.097994,1.050984,0.891981,0.865851,1.350282,0.379360
169,2015-08-14 15:00:00+00:00,1.099205,0.000000,0.000000,0,0,-7.071068e-01,-0.707107,-0.900969,-0.433884,...,1.120670,1.101614,0.863112,1.109451,1.098180,1.054841,0.896381,0.869033,1.343178,0.383047
170,2015-08-14 16:00:00+00:00,1.073096,2.638756,0.693147,0,0,-5.000000e-01,-0.866025,-0.900969,-0.433884,...,1.110952,1.101740,0.861248,1.099064,1.094321,1.056302,0.900131,0.871811,1.335536,0.386960
171,2015-08-14 17:00:00+00:00,1.033184,5.380558,2.079442,0,0,-2.588190e-01,-0.965926,-0.900969,-0.433884,...,1.094583,1.098540,0.859146,1.080241,1.084915,1.054452,0.902953,0.874003,1.327159,0.391133
172,2015-08-14 18:00:00+00:00,1.057790,5.303305,1.386294,0,0,-1.836970e-16,-1.000000,-0.900969,-0.433884,...,1.082314,1.097391,0.857191,1.073827,1.080742,1.054719,0.906235,0.876496,1.319046,0.395336


In [8]:
# Start an MLflow run for each set of parameters
run_count = 0
early_stopping = EarlyStopping(monitor='loss', min_delta=0.005, patience=5, verbose=1, mode='min')


for params in grid:
    with mlflow.start_run():
        clear_output(wait=True)
        print(run_count)
        # log the currently used parameters
        mlflow.log_params(params)

        # logging training size 
        mlflow.log_param("training data size", len(training))
        mlflow.log_param("unseen test data size", len(test))

        lookback = params['lookback']
        features = params['features']
        n_features = len(features)   

        # setting up model      
        # default LSTM-Design 
        model_layers_config = [
            {'type': 'LSTM', 'units': lookback * n_features, 'return_sequences': True, 'input_shape': (lookback, n_features)},
            {'type': 'Dropout', 'rate': 0.2},
            {'type': 'LSTM', 'units': lookback * n_features, 'return_sequences': False},
            {'type': 'Dropout', 'rate': 0.2}, 
            {'type': 'Dense', 'units': 1, 'activation': None}
        ]

        mlflow.log_param("model_layers_config", model_layers_config)

        model = create_lstm_model(model_layers_config)
        model.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=params['loss'], metrics=params['metrics'])

        # prep data
        X, y = create_sequences_np(data=training, features=features, lookback=lookback, datatype=params['datatype'])

        training_validation_split = 0.8
        split_idx = int(len(X) * training_validation_split)

        mlflow.log_param("training_validation_split", training_validation_split)

        # validation set with the last 20 percent of the dataset 
        X_train, X_val = X[:split_idx], X[split_idx:]
        y_train, y_val = y[:split_idx], y[split_idx:]

        mlflow.log_param("X_train.shape", X_train.shape)
        mlflow.log_param("y_train.shape", y_train.shape)

        mlflow.log_param("X_val.shape", X_val.shape)
        mlflow.log_param("y_val.shape", y_val.shape)
        
        # train model
        model.fit(X_train, y_train, epochs=params['epochs'], batch_size=params['batch_size'], validation_data=(X_val, y_val), callbacks=[early_stopping]) # , callbacks=[early_stopping]
        
        # Evaluate your model
        # Validation Data
        y_pred = model.predict(X_val)

        y_val_true = np.expm1(y_val)
        y_pred_true = np.expm1(y_pred)

        fig, ax = plt.subplots(figsize=(15, 7))
        ax.plot(y_val_true, label='Actual Values')
        ax.plot(y_pred_true, label='Predicted Values')
        ax.set_title('LSTM Model Validation Plot')
        ax.set_xlabel('Time')
        ax.set_ylabel('Close Price')
        ax.set_yscale('log')
        ax.legend()
        fig.tight_layout()
        artifact_path = 'plots'
        mlflow.log_figure(fig, artifact_path + '/validation_plot.png')
        plt.close(fig)

        # Unseen Data
        predicted_prices = []
        actual_prices = test['close_price_true'].values[lookback:]  # actual prices without log transformation
        timestamps = test['bucket'].values[lookback:]  # corresponding timestamps

        for i in range(lookback, len(test)):
            last_sequence = test.iloc[i-lookback:i][features].values.reshape((1, lookback, len(features)))
            last_sequence = np.array(last_sequence).astype(np.float16)
            predicted_log_price = model.predict(last_sequence)
            predicted_price = np.expm1(predicted_log_price)[0, 0]  # inverse log transformation
            predicted_prices.append(predicted_price)
        predicted_prices = np.array(predicted_prices)

        fig, ax = plt.subplots(figsize=(15, 7))
        ax.plot(timestamps, actual_prices, label='Actual Prices', color='blue')
        ax.plot(timestamps, predicted_prices, label='Predicted Prices', color='orange')
        ax.set_title('LSTM Model Predictions vs Actual Prices')
        ax.set_xlabel('Date/Time')
        ax.set_ylabel('Close Price')
        labels = ax.get_xticklabels()
        #ax.set_xticklabels(labels, rotation=45)
        ax.legend()
        fig.tight_layout()
        artifact_path = 'plots'
        mlflow.log_figure(fig, artifact_path + '/unseen_test_plot.png')
        plt.close(fig)

        # calculate some metrics 
        mse_value = mean_squared_error(actual_prices, predicted_prices)
        mae_value = mean_absolute_error(actual_prices, predicted_prices)
        rmse_value = np.sqrt(mse_value)
        mape_value = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100
        r2_value = r2_score(actual_prices, predicted_prices)
        explained_variance = explained_variance_score(actual_prices, predicted_prices)

        mlflow.log_metric('mse', mse_value)
        mlflow.log_metric('mae', mae_value)
        mlflow.log_metric('rmse', rmse_value)
        mlflow.log_metric('mape', mape_value)
        mlflow.log_metric('r2', r2_value)
        mlflow.log_metric('explained_variance', explained_variance)

        mlflow.sklearn.log_model(model, "model")

        run_count = run_count + 1


0


2023-12-03 12:51:53.098860: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-12-03 12:51:53.099047: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-12-03 12:51:53.099169: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Epoch 1/50


2023-12-03 12:51:55.756773: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 61371904 exceeds 10% of free system memory.
2023-12-03 12:51:55.822741: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 61371904 exceeds 10% of free system memory.
2023-12-03 12:52:00.906320: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2023-12-03 12:52:02.942170: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f3f61a4b310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-12-03 12:52:02.942199: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1060 6GB, Compute Capability 6.1
2023-12-03 12:52:02.946239: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-12-03 12:52:03.057304: I ./tensorflow/compiler/jit/device_comp

3526/3526 [==============================] - 38s 9ms/step - loss: 0.2169 - mean_absolute_error: 0.2169 - mean_squared_error: 0.0944 - accuracy: 0.0000e+00 - val_loss: 0.1964 - val_mean_absolute_error: 0.1964 - val_mean_squared_error: 0.0422 - val_accuracy: 0.0000e+00
Epoch 2/50
3526/3526 [==============================] - 29s 8ms/step - loss: 0.1695 - mean_absolute_error: 0.1695 - mean_squared_error: 0.0503 - accuracy: 0.0000e+00 - val_loss: 0.0780 - val_mean_absolute_error: 0.0780 - val_mean_squared_error: 0.0081 - val_accuracy: 0.0000e+00
Epoch 3/50
3526/3526 [==============================] - 29s 8ms/step - loss: 0.1549 - mean_absolute_error: 0.1549 - mean_squared_error: 0.0430 - accuracy: 0.0000e+00 - val_loss: 0.0832 - val_mean_absolute_error: 0.0832 - val_mean_squared_error: 0.0092 - val_accuracy: 0.0000e+00
Epoch 4/50
3526/3526 [==============================] - 31s 9ms/step - loss: 0.1433 - mean_absolute_error: 0.1433 - mean_squared_error: 0.0372 - accuracy: 0.0000e+00 - val_lo

/home/rileydavid/.local/lib/python3.10/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
